# 04 — Central-Chile megadrought: a climatology-relative view

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/04_megadrought_central_chile.ipynb)

**What we're investigating.** Since 2010 central Chile (30–40°S) has
lived through an uninterrupted sequence of dry years — the *Megadrought*
— driven by a mix of natural oceanic forcing and anthropogenic climate
change. Leveraging the continuous 30-year record, we characterise the
event as a **departure from the full-record (1990–2019) climatological
precipitationshed**: where did the moisture that rains on central Chile
evaporate from, and how did that supply change during the drought decade?

**What you'll get.** This notebook reproduces manuscript **FIG5**, a
single composite figure:

- **(a) Three maps** — the 30-year mean precipitationshed (1990–2019),
  and the departures of the historical baseline (1990–2009) and of the
  megadrought decade (2010–2019) from that climatology. The two
  departure panels share a symmetric diverging scale, so *blue* = a
  below-normal moisture source.
- **(b) Annual supply** — moisture partitioned into continental (land,
  green) and oceanic (Pacific, blue) sources as stacked bars
  (mm yr⁻¹), overlaid with the interannual continental recycling ratio
  (dark-red line) and the baseline / drought-period means (dotted).

**The takeaway.** The Pacific Ocean dominates the budget throughout,
and the continental recycling ratio stays essentially unchanged (~28%)
between the two periods — the drought reflects a reduction in the
*total* moisture supply, not a shift in the balance between oceanic
advection and continental recycling.

**Dataset.** AguaTrack-ARCO-SA **yearly aggregate** zarr store
(`AguaTrack_ARCO_SA_yearly.zarr`, 30 annual time steps). We sum the
tracked-evaporation source field over all tags inside the central-Chile
receptor box (30–38°S, 75–68°W).

**How to cite.** See the [repo README](https://github.com/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial#how-to-cite).

## Step 1 — Configuration

Everything you might want to edit lives in this single cell:

- **HuggingFace dataset** — `AguaTrackSA/AguaTrack-ARCO-SA-Aggregated`,
  the consolidated yearly zarr that holds all 30 years on one `time`
  axis. To run against a **local mirror** instead, point
  `AGUATRACK_YEARLY_URL` at the filesystem path of the store (e.g.
  `/path/to/AguaTrack_ARCO_SA_yearly.zarr`) and set `HF_REVISION = None`
  so `storage_options` is dropped.
- **Receptor box** — central Chile (~Santiago south to Puerto Montt).
  Set `LAT_MIN`/`LAT_MAX`/`LON_MIN`/`LON_MAX` to any other South
  American sink to compare drought-decade source shifts there.
- **Period split** — 20 years baseline + 10 years drought, the standard
  split used by the Chilean climate-science community for this event.

In [ ]:
HF_REVISION = "main"

LAT_MIN, LAT_MAX = -38.0, -30.0
LON_MIN, LON_MAX = -75.0, -68.0
HISTORICAL_YEARS = list(range(1990, 2010))   # 20 years baseline
DROUGHT_YEARS = list(range(2010, 2020))      # 10 years drought decade
ALL_YEARS = HISTORICAL_YEARS + DROUGHT_YEARS

AGUATRACK_YEARLY_URL = (
    "hf://datasets/AguaTrackSA/AguaTrack-ARCO-SA-Aggregated"
    "/AguaTrack_ARCO_SA_yearly.zarr"
)

## Step 2 — Install dependencies (Colab only)

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask",
    )

## Step 3 — Imports and plotting style

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import xarray as xr

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
})

## Step 4 — Find tag cells inside the receptor box

Each `tagging_mask` index is one real grid cell. We keep only the tags
whose `(tag_lat, tag_lon)` fall inside the central-Chile receptor box —
summing over them later collapses all of central Chile into a single
moisture sink.

In [ ]:
open_kwargs = {} if HF_REVISION is None else {"storage_options": {"revision": HF_REVISION}}
ds_full = xr.open_zarr(AGUATRACK_YEARLY_URL, **open_kwargs)

in_box = (
    (ds_full.tag_lat >= LAT_MIN) & (ds_full.tag_lat <= LAT_MAX)
    & (ds_full.tag_lon >= LON_MIN) & (ds_full.tag_lon <= LON_MAX)
)
box_tag_idx = np.flatnonzero(in_box.values)
print(f"tag cells in receptor box: {len(box_tag_idx)}")

## Step 5 — Build the per-year source maps and land/ocean budget

Pangeo idiom: pre-select the receptor-box tags **first** (the fast chunk
axis), rename the yearly `time` axis to an integer `year` coordinate,
then sum over the tags to get one 2-D source map per year. We
materialise the summed source field once, and derive the land-only sum
using the store's explicit `lsm` land-sea mask; the ocean sum is the
residual, and the continental recycling ratio is land / total.

In [ ]:
ds_all = (
    ds_full.isel(tagging_mask=box_tag_idx)
    .assign_coords(year=("time", ds_full.time.dt.year.values))
    .swap_dims({"time": "year"})
    .drop_vars("time")
)
lsm = ds_full.lsm

# Per-year 2-D source map (year, lat, lon), summed over the box tags.
year_src = ds_all.e_track.sum("tagging_mask").load()
year_total = year_src.sum(("latitude", "longitude"))
year_land = year_src.where(lsm).sum(("latitude", "longitude"))
ds_full.close()

year_ocean = year_total - year_land
year_ratio = year_land / year_total.where(year_total > 0)

print(f"loaded {year_src.sizes['year']} years "
      f"({year_src.nbytes / 1e6:.1f} MB in RAM)")

## Step 6 — Climatology and period departures

The 30-year mean is the climatological precipitationshed. Each period's
departure is `period mean − 30-yr climatology`. The two departure maps
share a symmetric diverging scale centred on zero so the drought's
below-normal moisture supply (blue) is directly comparable to the
slightly above-normal baseline.

In [ ]:
clim_map = year_src.sel(year=ALL_YEARS).mean("year")                 # 30-yr climatology
hist_anom = year_src.sel(year=HISTORICAL_YEARS).mean("year") - clim_map
drought_anom = year_src.sel(year=DROUGHT_YEARS).mean("year") - clim_map

vmax = float(clim_map.max())
levels = np.linspace(0, vmax, 11)
# Shared symmetric scale for the two departure panels.
adiff = float(max(abs(float(hist_anom.min())), abs(float(hist_anom.max())),
                  abs(float(drought_anom.min())), abs(float(drought_anom.max()))))
alevels = np.linspace(-adiff, adiff, 11)

print(f"climatology max={vmax:.1f} mm/yr  departure scale=±{adiff:.1f} mm/yr")

## Step 7 — Compose FIG5

Panel (a) is the three maps (climatology + two departures), zoomed to
central-south Chile and its SE-Pacific source region so the receptor
box (red) is the visual focus. Panel (b) is the annual stacked
continental/oceanic budget with the recycling-ratio line and dotted
period means.

In [ ]:
all_yrs = year_src.year.values
land_arr, ocean_arr = year_land.values, year_ocean.values
ratio_arr = (year_ratio * 100).values
y_max = float((year_land + year_ocean).max())
hist_mean = float((year_ratio.sel(year=HISTORICAL_YEARS) * 100).mean())
drought_mean = float((year_ratio.sel(year=DROUGHT_YEARS) * 100).mean())

fig = plt.figure(figsize=(12, 11))
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 0.92], hspace=0.42)
gs_top = gs[0].subgridspec(1, 3, wspace=0.06)
axm = [fig.add_subplot(gs_top[i], projection=ccrs.PlateCarree()) for i in range(3)]


def style_map(ax, title):
    ax.coastlines(resolution="50m", color="black", linewidth=0.8)
    # Zoomed to central-south Chile + its SE-Pacific source region so the
    # departure signal in the receptor box is the visual focus. Clamped to
    # the data's western edge (-90) so no empty tracking area is shown.
    ax.set_extent([-90, -56, -51, -19], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.0)
    gl.top_labels = gl.right_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 10))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 10))
    gl.xlabel_style = gl.ylabel_style = {"size": 15}
    ax.set_title(title)


kw = dict(transform=ccrs.PlateCarree(), add_colorbar=False)
cf1 = clim_map.plot.contourf(ax=axm[0], levels=levels, cmap=cmc.batlowW_r, extend="max", **kw)
cf2 = hist_anom.plot.contourf(ax=axm[1], levels=alevels, cmap="RdBu_r", extend="both", **kw)
drought_anom.plot.contourf(ax=axm[2], levels=alevels, cmap="RdBu_r", extend="both", **kw)
for ax in axm:
    ax.add_patch(mpatches.Rectangle(
        (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
        linewidth=2, edgecolor="red", facecolor="none",
        transform=ccrs.PlateCarree(), zorder=5))
style_map(axm[0], "Climatology\n1990–2019")
style_map(axm[1], "Baseline anomaly\n1990–2009")
style_map(axm[2], "Drought anomaly\n2010–2019")

cb1 = fig.colorbar(cf1, ax=axm[0], orientation="horizontal", fraction=0.05, pad=0.10, aspect=20)
cb1.set_label("Moisture Contribution (mm/yr)")
cb1.ax.tick_params(labelsize=15)
cb2 = fig.colorbar(cf2, ax=[axm[1], axm[2]], orientation="horizontal", fraction=0.05, pad=0.10, aspect=40)
cb2.set_label("Departure from climatology (mm/yr)")
cb2.ax.tick_params(labelsize=15)

# ---- Panel (b): annual stacked bar + recycling ratio ----
axb = fig.add_subplot(gs[1])
axb.axvspan(1989.5, 2009.5, alpha=0.07, color="steelblue")
axb.axvspan(2009.5, 2019.5, alpha=0.07, color="firebrick")
axb.axvline(2009.5, color="black", lw=1.2, linestyle="--")
axb.bar(all_yrs, land_arr, color="#4a9e6b", alpha=0.88, label="Continental (land)")
axb.bar(all_yrs, ocean_arr, bottom=land_arr, color="#3a7ebf", alpha=0.88, label="Oceanic (Pacific)")
axb.text(1999.5, y_max * 0.99, "Baseline\n1990–2009", ha="center", va="top",
         color="steelblue", fontweight="bold")
axb.text(2014.5, y_max * 0.99, "Megadrought\n2010–2019", ha="center", va="top",
         color="firebrick", fontweight="bold")
axb.set_xlabel("Year")
axb.set_ylabel("Moisture Contribution (mm/yr)")
axb.legend(loc="upper left")
axb.set_xlim(1989.5, 2019.5)

axr = axb.twinx()
axr.plot(all_yrs, ratio_arr, color="darkred", lw=2, marker="o", ms=4)
axr.set_ylabel("Continental Recycling Ratio (%)", color="darkred")
axr.tick_params(axis="y", labelcolor="darkred")
axr.set_ylim(0, max(float(np.nanmax(ratio_arr)) * 1.35, 10.0))
n_hist, n_total = len(HISTORICAL_YEARS), len(ALL_YEARS)
axr.axhline(hist_mean, xmin=0.0, xmax=n_hist / n_total, color="darkred", lw=1.4, linestyle=":")
axr.axhline(drought_mean, xmin=n_hist / n_total, xmax=1.0, color="darkred", lw=1.4, linestyle=":")
bbox_lbl = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8)
axr.annotate(f"baseline mean {hist_mean:.1f}%", xy=(1991, hist_mean), xytext=(0, 8),
             textcoords="offset points", color="darkred", va="bottom", ha="left",
             fontsize=13, bbox=bbox_lbl)
axr.annotate(f"drought mean {drought_mean:.1f}%", xy=(2010.5, drought_mean), xytext=(0, -18),
             textcoords="offset points", color="darkred", va="top", ha="left",
             fontsize=13, bbox=bbox_lbl)

fig.text(0.02, 0.965, "(a)", fontsize=20, fontweight="bold")
fig.text(0.02, 0.47, "(b)", fontsize=20, fontweight="bold")

OUT = Path("outputs/megadrought"); OUT.mkdir(parents=True, exist_ok=True)
out = OUT / "FIG5_megadrought_climrel.png"
fig.savefig(out, bbox_inches="tight", dpi=200)
plt.show()
print(f"saved {out}")

## Step 8 — Summary

The continental recycling ratio barely moves between the baseline and
the drought decade, confirming that the megadrought is a deficit in the
*total* moisture supply rather than a change in the oceanic-vs-recycled
balance.

In [ ]:
print("Central Chile megadrought (30–38°S receptor box):\n")
print(f"  Continental recycling ratio : baseline {hist_mean:.1f}%  ->  drought {drought_mean:.1f}%  "
      f"({drought_mean - hist_mean:+.1f} pp)")
print(f"  Climatological source max   : {vmax:.1f} mm/yr")
print(f"  Departure scale             : ±{adiff:.1f} mm/yr")